# Stage 05 - Train a Readiness Model

Train and track a reproducible scikit-learn candidate using the governed readiness feature frame.

> Model output prioritizes human review; it does not authorize readiness or safety decisions.

In [ ]:
from contextlib import nullcontext
from pathlib import Path
import importlib.util
import json

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILE = "readiness_observation_features.csv"
FEATURE_TABLE = "silver_readiness_feature_store"
DATA_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_FILE,
    Path("../data") / DATA_FILE,
    Path("data") / DATA_FILE,
    Path("Files") / DATA_FILE,
]


def locate_data_file(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {DATA_FILE}. Checked: {checked}")


def min_max_scale(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / span

spark_session = globals().get("spark")
if spark_session is not None and spark_session.catalog.tableExists(FEATURE_TABLE):
    frame = spark_session.table(FEATURE_TABLE).toPandas()
    data_path = f"Lakehouse table {FEATURE_TABLE}"
else:
    data_path = locate_data_file(DATA_CANDIDATES)
    frame = pd.read_csv(data_path)

frame["feature_timestamp_utc"] = pd.to_datetime(frame["feature_timestamp_utc"], utc=True)
frame = frame.sort_values(
    ["scenario_id", "simulation_run_id", "system_instance_id", "feature_timestamp_utc"]
).reset_index(drop=True)
frame["quality_gap"] = 1.0 - frame["quality_rate"]
frame["feature_recency_minutes"] = frame["track_freshness_seconds"] / 60.0
frame["health_decline_flag"] = (frame["health_trend_index"] < 0).astype(int)
frame["maintenance_age_band_index"] = frame["maintenance_age_category"].map(
    {"fresh-service": 0, "steady-cycle": 1, "extended-cycle": 2}
).astype(int)
frame["run_quality_delta"] = frame["quality_rate"] - frame.groupby("simulation_run_id")["quality_rate"].transform("mean")
gap_totals = frame.groupby("simulation_run_id")["gap_count"].transform("sum").replace(0, 1)
frame["run_gap_share"] = frame["gap_count"] / gap_totals
frame["baseline_alignment_gap"] = frame["baseline_deviation_index"] + frame["quality_gap"]
frame["deterministic_baseline_score"] = (
    0.30 * min_max_scale(frame["track_freshness_seconds"])
    + 0.20 * min_max_scale(frame["gap_count"])
    + 0.20 * min_max_scale(frame["quality_gap"])
    + 0.15 * min_max_scale(frame["abstract_ack_lag_seconds"])
    + 0.10 * min_max_scale(frame["baseline_deviation_index"])
    + 0.05 * min_max_scale(frame["maintenance_age_days"])
)

feature_columns = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'maintenance_age_category', 'test_phase', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
numeric_features = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
categorical_features = ["maintenance_age_category", "test_phase"]
lineage_columns = ['classification', 'scenario_id', 'simulation_run_id', 'test_event_id', 'track_id', 'site_id', 'system_instance_id', 'system_family', 'feature_timestamp_utc', 'source_snapshot_id', 'feature_snapshot_id', 'baseline_snapshot_id', 'finding_snapshot_id']
name_flags = [column for column in feature_columns if any(token in column for token in ("label", "outcome", "decision", "pass", "fail"))]
exact_match_flags = [column for column in feature_columns if frame[column].astype(str).equals(frame["synthetic_review_priority_label"].astype(str))]
assert not set(feature_columns) & set(lineage_columns)
assert not name_flags, f"Potential leakage by feature name: {name_flags}"
assert not exact_match_flags, f"Potential leakage by exact feature duplication: {exact_match_flags}"

run_order = frame.groupby("simulation_run_id")["feature_timestamp_utc"].min().sort_values().index.tolist()
train_runs = run_order[:2]
validation_runs = run_order[2:3]
test_runs = run_order[3:]

train_frame = frame[frame["simulation_run_id"].isin(train_runs)].copy()
validation_frame = frame[frame["simulation_run_id"].isin(validation_runs)].copy()
test_frame = frame[frame["simulation_run_id"].isin(test_runs)].copy()

X_train = train_frame[feature_columns]
y_train = train_frame["synthetic_review_priority_label"]
X_validation = validation_frame[feature_columns]
y_validation = validation_frame["synthetic_review_priority_label"]
X_test = test_frame[feature_columns]
y_test = test_frame["synthetic_review_priority_label"]

MLFLOW_AVAILABLE = importlib.util.find_spec("mlflow") is not None
print(json.dumps({
    "data_path": str(data_path),
    "train_runs": train_runs,
    "validation_runs": validation_runs,
    "test_runs": test_runs,
    "mlflow_available": MLFLOW_AVAILABLE,
}, indent=2))


In [ ]:
def prevalence_rank(probabilities, positive_rate):
    top_n = max(1, int(round(len(probabilities) * positive_rate)))
    ranking = pd.Series(probabilities).rank(method="first", ascending=False)
    return (ranking <= top_n).astype(int)


def metric_row(model_name, split_name, y_true, probabilities, positive_rate):
    y_true = pd.Series(y_true).astype(int)
    predicted = prevalence_rank(probabilities, positive_rate)
    return {
        "model_name": model_name,
        "split": split_name,
        "roc_auc": round(float(roc_auc_score(y_true, probabilities)), 4),
        "average_precision": round(float(average_precision_score(y_true, probabilities)), 4),
        "brier_loss": round(float(brier_score_loss(y_true, probabilities)), 4),
        "predicted_priority_rate": round(float(predicted.mean()), 4),
        "observed_priority_rate": round(float(y_true.mean()), 4),
    }


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            categorical_features,
        ),
    ]
)

model_specs = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=7),
    "random_forest": RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=7),
}

train_positive_rate = float(y_train.mean())
results = []
fitted_models = {}
mlflow_run_ids = {}
mlflow = None
if MLFLOW_AVAILABLE:
    import mlflow

for model_name, estimator in model_specs.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    run_context = mlflow.start_run(run_name=f"demo04_{model_name}", nested=True) if mlflow is not None else nullcontext()
    with run_context as active_run:
        pipeline.fit(X_train, y_train)
        fitted_models[model_name] = pipeline
        if active_run is not None:
            mlflow_run_ids[model_name] = active_run.info.run_id
            mlflow.log_params({
                "model_name": model_name,
                "train_runs": ",".join(train_runs),
                "validation_runs": ",".join(validation_runs),
                "test_runs": ",".join(test_runs),
                "feature_count": len(feature_columns),
            })
        for split_name, X_split, y_split in [
            ("validation", X_validation, y_validation),
            ("test", X_test, y_test),
        ]:
            probabilities = pipeline.predict_proba(X_split)[:, 1]
            row = metric_row(model_name, split_name, y_split, probabilities, train_positive_rate)
            results.append(row)
            if active_run is not None:
                mlflow.log_metric(f"{split_name}_roc_auc", row["roc_auc"])
                mlflow.log_metric(f"{split_name}_average_precision", row["average_precision"])
                mlflow.log_metric(f"{split_name}_brier_loss", row["brier_loss"])

for split_name, split_frame in [("validation", validation_frame), ("test", test_frame)]:
    results.append(
        metric_row(
            "deterministic_baseline",
            split_name,
            split_frame["synthetic_review_priority_label"],
            split_frame["deterministic_baseline_score"],
            train_positive_rate,
        )
    )

results_df = pd.DataFrame(results).sort_values(["split", "roc_auc"], ascending=[True, False]).reset_index(drop=True)
results_df


In [ ]:
validation_ranking = results_df[results_df["split"] == "validation"].sort_values("roc_auc", ascending=False).reset_index(drop=True)
selected_sklearn_model = validation_ranking.iloc[0]["model_name"]

model_validation_summary = {
    "classification": "SYNTHETIC_UNCLASS",
    "data_path": str(data_path),
    "data_lineage": {
        "source_snapshot_ids": sorted(frame["source_snapshot_id"].unique().tolist()),
        "feature_snapshot_ids": sorted(frame["feature_snapshot_id"].unique().tolist()),
        "baseline_snapshot_ids": sorted(frame["baseline_snapshot_id"].unique().tolist()),
        "finding_snapshot_ids": sorted(frame["finding_snapshot_id"].unique().tolist()),
    },
    "split_strategy": {
        "group_key": "simulation_run_id",
        "train_runs": train_runs,
        "validation_runs": validation_runs,
        "test_runs": test_runs,
    },
    "candidate_features": feature_columns,
    "deterministic_baseline_note": "Invented demo weighting over freshness, gaps, quality gap, acknowledgement lag, deviation, and maintenance age.",
    "leakage_checks": {
        "excluded_lineage_columns": sorted(lineage_columns),
        "feature_name_flags": name_flags,
        "exact_match_flags": exact_match_flags,
    },
    "selected_sklearn_model": selected_sklearn_model,
    "mlflow": {
        "available": MLFLOW_AVAILABLE,
        "run_ids": mlflow_run_ids,
    },
    "limitations": [
        "The label is synthetic and supports analyst prioritization only.",
        "Only four synthetic runs are available, so run-level holdout matters more than leaderboard metrics.",
        "No official pass or fail outcome is created by this notebook.",
    ],
}

print(json.dumps(model_validation_summary, indent=2))
results_df
